In [1]:
from __future__ import annotations

import sys
from pathlib import Path
import time

project_dir = Path.cwd()

# Modified .py files can be placed either in the same folder as this notebook
# or inside a subfolder named FHF_modified.
module_dir = project_dir
if not (module_dir / "FHF_runner.py").exists():
    candidate = project_dir / "FHF_modified"
    if (candidate / "FHF_runner.py").exists():
        module_dir = candidate

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from FHF_runner import Args, set_seed, run_fhf
from FHF_nf import load_clients_from_csv
from FHF_Client import Client

import warnings
warnings.filterwarnings("ignore", message=".*flash attention.*")

In [2]:
print("torch version:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("working directory:", project_dir)
print("module directory:", module_dir)

torch version: 2.3.1+cu118
device: cuda
working directory: E:\yjz\FHF\FHF_ausgrid2013
module directory: E:\yjz\FHF\FHF_ausgrid2013


In [3]:
# -------------------------
# Parameter setup
# -------------------------
args = Args()
args.seed = 42
set_seed(args.seed, torch_num_threads=getattr(args, "torch_num_threads", 1))
args.device = "cuda" if torch.cuda.is_available() else "cpu"

args.csv_path = "E:/yjz/Datasets/AusGrid2013/AusGrid2013_univariate.csv"
#"E:/yjz/Datasets/AusGrid2013/AusGrid2013_univariate.csv"
args.model_type = "lstm"
args.truncated = None
args.fh = 0
args.lags = 48
args.num_rounds = 100 + args.fh * 50

args.recon_method = [
    "bu",
    "mint_cov",
    "mint_shrinkage",
    "mint_ols",
    "mint_var",
    "wls_structure",
]

# -------------------------
# Early stopping settings
# -------------------------
args.early_stop_enabled = True
args.early_stop_tol = 1e-5
args.early_stop_metric = "avg_normalised_loss_delta"
args.min_rounds = 20 + args.fh * 50

# Logging / output format
args.verbose_rounds = True
args.print_every = 1
args.return_result = True

args


Args(csv_path='E:/yjz/Datasets/AusGrid2013/AusGrid2013_univariate.csv', partition_col='partition_id', series_col='unique_id', time_col='ds', target_col='y', truncated=None, lags=48, fh=0, test_ratio=0.2, batch_size=256, model_type='lstm', input_size=1, hidden_size=64, num_layers=2, dropout=0.1, c_in=1, d_model=128, n_heads=4, e_layers=3, device='cuda', num_rounds=100, local_epochs=1, lr=0.001, weight_decay=0.0, optimizer='adam', recon_method=['bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure'], ridge=1e-06, td_eps=1e-08, seed=42, validate_p2p=True, p2p_atol=1e-08, torch_num_threads=1, early_stop_enabled=True, early_stop_tol=1e-05, early_stop_metric='avg_normalised_loss_delta', min_rounds=20, verbose_rounds=True, print_every=1, return_result=True)

In [4]:
# Optional quick data check before training
client_ids, cid2series_preview, cid2df_preview = load_clients_from_csv(
    csv_path=args.csv_path,
    partition_col=args.partition_col,
    series_col=args.series_col,
    time_col=args.time_col,
    target_col=args.target_col,
    lags=args.lags,
    fh=args.fh,
    truncated=args.truncated,
)

c = Client(client_ids[0], cid2series_preview[client_ids[0]], cid2df_preview[client_ids[0]], args)
c.prepare_data()

print("n_clients:", len(client_ids))
print("first client:", client_ids[0], cid2series_preview[client_ids[0]])
print("device:", c.device)
print("train samples:", c.num_train_samples(), "batches:", len(c.train_loader))

n_clients: 400
first client: 0 GC
device: cuda
train samples: 13968 batches: 55


In [5]:
# -------------------------
# Run FHF
# -------------------------
result = run_fhf(args)

clients = result["clients"]
cloud = result["cloud"]
cid2series = result["cid2series"]
timings = result["timings"]
round_logs = result["round_logs"]
early_stop = result["early_stop"]

print("early_stop:", early_stop)

Loaded 400 clients.
Recon methods: ['bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure']
[Round 1/100] avg_normalised_loss=0.566015, avg_actual_loss=0.648553, delta=nan, stopped=False
[Round 2/100] avg_normalised_loss=0.451806, avg_actual_loss=0.295206, delta=0.114209, stopped=False
[Round 3/100] avg_normalised_loss=0.444286, avg_actual_loss=0.232257, delta=0.00751947, stopped=False
[Round 4/100] avg_normalised_loss=0.443271, avg_actual_loss=0.236471, delta=0.00101527, stopped=False
[Round 5/100] avg_normalised_loss=0.442237, avg_actual_loss=0.217554, delta=0.00103418, stopped=False
[Round 6/100] avg_normalised_loss=0.441079, avg_actual_loss=0.220718, delta=0.00115801, stopped=False
[Round 7/100] avg_normalised_loss=0.440442, avg_actual_loss=0.21288, delta=0.000636694, stopped=False
[Round 8/100] avg_normalised_loss=0.439335, avg_actual_loss=0.210752, delta=0.00110701, stopped=False
[Round 9/100] avg_normalised_loss=0.438234, avg_actual_loss=0.217792, delta=0.00

In [6]:
# Cloud-round training log
round_log_df = pd.DataFrame(round_logs)
round_log_df.tail()

,round,num_rounds,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,avg_train_loss,n_train_nodes,early_stop_enabled,early_stop_tol,early_stop_metric,stopped
67,68,100,0.390593,0.152097,0.000060,0.390593,400,True,0.00001,avg_normalised_loss_delta,False
68,69,100,0.390758,0.152569,0.000164,0.390758,400,True,0.00001,avg_normalised_loss_delta,False
69,70,100,0.390091,0.152296,0.000666,0.390091,400,True,0.00001,avg_normalised_loss_delta,False
70,71,100,0.389549,0.151550,0.000542,0.389549,400,True,0.00001,avg_normalised_loss_delta,False
71,72,100,0.389559,0.150628,0.000010,0.389559,400,True,0.00001,avg_normalised_loss_delta,True


In [7]:
# Check reconciled forecast methods and shapes
cid0 = list(clients.keys())[0]
print("client:", cid0, cid2series[cid0])
print("methods:", list(clients[cid0].reconciled_forecasts.keys()))

for method, arr in clients[cid0].reconciled_forecasts.items():
    print(method, np.asarray(arr).shape)

client: 0 GC
methods: ['bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure']
bu (3504, 1)
mint_cov (3504, 1)
mint_shrinkage (3504, 1)
mint_ols (3504, 1)
mint_var (3504, 1)
wls_structure (3504, 1)


## Level-wise evaluation: MAE, MASE, RMSSE

In [8]:
def _as_2d(arr):
    arr = np.asarray(arr, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr[:, None]
    return arr


def _target_cols(args):
    return ['y'] + [f'post_{i}' for i in range(1, int(args.fh) + 1)]


def split_node_df(df_one, args):
    time_col = getattr(args, 'time_col', 'ds')
    sub = df_one.sort_values(time_col).copy().reset_index(drop=True)

    n_windows = len(sub)
    n_raw = n_windows + int(args.lags) + int(args.fh)
    n_train_raw = int((1 - float(args.test_ratio)) * n_raw)
    n_train_windows = n_train_raw - int(args.lags) - int(args.fh)

    if n_train_windows <= 0 or n_train_windows >= n_windows:
        raise ValueError(
            f'Bad split: n_windows={n_windows}, n_train_windows={n_train_windows}'
        )

    return sub.iloc[:n_train_windows].copy(), sub.iloc[n_train_windows:].copy()


def make_naive_forecast(node, args):
    """Simple random-walk naive forecast: use the latest available lag, lags_1.

    For multi-step horizons, the same last observed value is repeated across
    horizons. This is the standard persistence baseline and uses no test future
    information.
    """
    _, ts_df = split_node_df(node.df, args)
    y_true = _as_2d(node.test_true)
    if 'lags_1' not in ts_df.columns:
        raise KeyError("Cannot compute naive forecast because 'lags_1' is missing.")
    naive_1d = ts_df['lags_1'].to_numpy(dtype=np.float64)
    if len(naive_1d) != y_true.shape[0]:
        raise ValueError(
            f'Naive length mismatch: naive={len(naive_1d)}, y_true={y_true.shape[0]}'
        )
    return np.repeat(naive_1d[:, None], y_true.shape[1], axis=1)


def _safe_scale_denominators(train_true, eps=1e-12):
    """Return MASE and RMSSE denominators from training true values only."""
    train_true = _as_2d(train_true)
    if train_true.shape[0] <= 1:
        return np.nan, np.nan

    diff = np.diff(train_true, axis=0)
    mase_denom = np.nanmean(np.abs(diff))
    rmsse_denom = np.nanmean(diff ** 2)

    if (not np.isfinite(mase_denom)) or mase_denom <= eps:
        mase_denom = np.nan
    if (not np.isfinite(rmsse_denom)) or rmsse_denom <= eps:
        rmsse_denom = np.nan

    return float(mase_denom), float(rmsse_denom)


def _role_from_level(level, min_level, max_level):
    if level == min_level:
        return 'top'
    if level == max_level:
        return 'bottom'
    return 'middle'


def compute_metrics_by_level(clients, cid2series, args, include_naive=True, eps=1e-12):
    rows = []

    levels = {
        int(cid): int(str(series).count('/') + 1)
        for cid, series in cid2series.items()
    }
    min_level = min(levels.values())
    max_level = max(levels.values())

    for cid in sorted(clients.keys()):
        node = clients[cid]
        unique_id = str(cid2series[cid])
        level = int(unique_id.count('/') + 1)
        role = _role_from_level(level, min_level, max_level)

        y_true = _as_2d(node.test_true)
        train_true = _as_2d(node.y2)
        mase_denom, rmsse_denom = _safe_scale_denominators(train_true, eps=eps)

        method_forecasts = {'base': node.base_forecast}
        if include_naive:
            method_forecasts['naive'] = make_naive_forecast(node, args)
        method_forecasts.update(node.reconciled_forecasts)

        for method, forecast in method_forecasts.items():
            y_pred = _as_2d(forecast)
            if y_pred.shape != y_true.shape:
                raise ValueError(
                    f'Shape mismatch for cid={cid}, method={method}: '
                    f'y_true={y_true.shape}, y_pred={y_pred.shape}'
                )

            error = y_true - y_pred
            mae = float(np.nanmean(np.abs(error)))
            mse = float(np.nanmean(error ** 2))
            mase = mae / mase_denom if np.isfinite(mase_denom) else np.nan
            rmsse = np.sqrt(mse / rmsse_denom) if np.isfinite(rmsse_denom) else np.nan

            rows.append({
                'node_id': int(cid),
                'unique_id': unique_id,
                'level': level,
                'role': role,
                'method': str(method),
                'MAE': mae,
                'MASE': float(mase) if np.isfinite(mase) else np.nan,
                'RMSSE': float(rmsse) if np.isfinite(rmsse) else np.nan,
            })

    per_node_metrics = pd.DataFrame(rows)

    metrics_by_level = (
        per_node_metrics
        .groupby(['level', 'role', 'method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['level', 'method'])
        .reset_index(drop=True)
    )

    overall_metrics = (
        per_node_metrics
        .groupby(['method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['method'])
        .reset_index(drop=True)
    )

    return per_node_metrics, metrics_by_level, overall_metrics


def build_forecast_table_fhf(clients, cid2series, args, include_naive=True):
    """Build a per-series forecast table with base, reconciled, and naive forecasts."""
    rows = []
    target_cols = _target_cols(args)
    time_col = getattr(args, 'time_col', 'ds')

    for cid in sorted(clients.keys()):
        node = clients[cid]
        _, ts_df = split_node_df(node.df, args)
        unique_id = str(cid2series[cid])
        y_true = _as_2d(node.test_true)
        base = _as_2d(node.base_forecast)
        naive = make_naive_forecast(node, args) if include_naive else None
        rec = {method: _as_2d(arr) for method, arr in node.reconciled_forecasts.items()}

        for h_idx, target_col in enumerate(target_cols):
            for t_idx, (_, row) in enumerate(ts_df.iterrows()):
                out = {
                    'node_id': int(cid),
                    'unique_id': unique_id,
                    'ds': row[time_col],
                    'horizon_index': int(h_idx),
                    'target_col': target_col,
                    'y_true': float(y_true[t_idx, h_idx]),
                    'base': float(base[t_idx, h_idx]),
                }
                if include_naive:
                    out['naive'] = float(naive[t_idx, h_idx])
                for method, arr in rec.items():
                    out[str(method)] = float(arr[t_idx, h_idx])
                rows.append(out)

    return pd.DataFrame(rows)


def make_timing_df(timings):
    ordered_modules = [
        'setup_sec',
        'data_preparation_sec',
        'model_server_initialization_sec',
        'fl_training_sec',
        'residual_collection_sec',
        'recon_matrix_sec',
        'p2p_reconcile_sec',
        'reconciliation_sec',
        'evaluation_sec',
        'total_sec',
    ]
    return pd.DataFrame(
        [{'module': key, 'seconds': timings.get(key, np.nan)} for key in ordered_modules]
    )


def save_fhf_outputs(output_prefix, clients, cid2series, args, per_node_metrics,
                     metrics_by_level, overall_metrics, round_logs, timings):
    """Save forecasts, per-series metrics, aggregate metrics, round logs, and time costs."""
    forecast_table = build_forecast_table_fhf(clients, cid2series, args, include_naive=True)
    timing_df = make_timing_df(timings)

    paths = {
        'forecasts': f'{output_prefix}_forecasts.csv',
        'per_node_metrics': f'{output_prefix}_per_node_metrics.csv',
        'metrics_by_level': f'{output_prefix}_metrics_by_level.csv',
        'overall_metrics': f'{output_prefix}_overall_metrics.csv',
        'round_logs': f'{output_prefix}_round_logs.csv',
        'timing': f'{output_prefix}_timing.csv',
    }

    forecast_table.to_csv(paths['forecasts'], index=False)
    per_node_metrics.to_csv(paths['per_node_metrics'], index=False)
    metrics_by_level.to_csv(paths['metrics_by_level'], index=False)
    overall_metrics.to_csv(paths['overall_metrics'], index=False)
    pd.DataFrame(round_logs).to_csv(paths['round_logs'], index=False)
    timing_df.to_csv(paths['timing'], index=False)

    return paths, forecast_table, timing_df


_eval_t0 = time.perf_counter()
per_node_metrics, metrics_by_level, overall_metrics = compute_metrics_by_level(
    clients, cid2series, args, include_naive=True
)
timings['evaluation_sec'] = time.perf_counter() - _eval_t0
timings['total_sec'] = (
    timings.get('setup_sec', 0.0)
    + timings.get('fl_training_sec', 0.0)
    + timings.get('reconciliation_sec', 0.0)
    + timings.get('evaluation_sec', 0.0)
)

output_prefix = f'fh{args.fh + 1}_FHF'
output_paths, forecast_table, timing_df = save_fhf_outputs(
    output_prefix=output_prefix,
    clients=clients,
    cid2series=cid2series,
    args=args,
    per_node_metrics=per_node_metrics,
    metrics_by_level=metrics_by_level,
    overall_metrics=overall_metrics,
    round_logs=round_logs,
    timings=timings,
)

result['timings'] = timings
result['per_node_metrics'] = per_node_metrics
result['metrics_by_level'] = metrics_by_level
result['overall_metrics'] = overall_metrics
result['forecast_table'] = forecast_table
result['output_paths'] = output_paths

print('Saved outputs:')
for name, path in output_paths.items():
    print(f'  {name}: {path}')

metrics_by_level


Saved outputs:
  forecasts: fh1_FHF_forecasts.csv
  per_node_metrics: fh1_FHF_per_node_metrics.csv
  metrics_by_level: fh1_FHF_metrics_by_level.csv
  overall_metrics: fh1_FHF_overall_metrics.csv
  round_logs: fh1_FHF_round_logs.csv
  timing: fh1_FHF_timing.csv


,level,role,method,MAE,MASE,RMSSE,n_nodes
0,1,top,base,7.441643,1.214632,1.106056,1
1,1,top,bu,7.473591,1.219847,1.158394,1
2,1,top,mint_cov,6.699960,1.093574,1.004601,1
3,1,top,mint_ols,7.434820,1.213519,1.105601,1
4,1,top,mint_shrinkage,6.742238,1.100475,1.004152,1
5,1,top,mint_var,7.329495,1.196327,1.123026,1
6,1,top,naive,7.763219,1.267120,1.198684,1
7,1,top,wls_structure,7.434820,1.213519,1.105601,1
8,2,middle,base,0.251599,1.122130,1.015859,100
9,2,middle,bu,0.249183,1.115523,1.015410,100


In [9]:
# Optional overall average across all hierarchy nodes
overall_metrics

,method,MAE,MASE,RMSSE,n_nodes
0,base,0.179055,1.139720,1.011875,400
1,bu,0.178531,1.138081,1.011893,400
2,mint_cov,0.179796,1.167740,1.011360,400
3,mint_ols,0.181474,1.165214,1.012731,400
4,mint_shrinkage,0.178783,1.158433,1.007972,400
5,mint_var,0.178825,1.142509,1.010952,400
6,naive,0.185562,1.167698,1.133205,400
7,wls_structure,0.181474,1.165214,1.012731,400


## Timing report

In [10]:
# Timing report saved by save_fhf_outputs(...).
timing_df


,module,seconds
0,setup_sec,58.700686
1,data_preparation_sec,58.623020
2,model_server_initialization_sec,0.077663
3,fl_training_sec,21477.638075
4,residual_collection_sec,129.613794
5,recon_matrix_sec,2.247081
6,p2p_reconcile_sec,22.524372
7,reconciliation_sec,154.385248
8,evaluation_sec,6.531974
9,total_sec,21697.255983


In [11]:
# Compact run summary
summary = {
    'stopped_early': result['stopped_early'],
    'stop_reason': result['stop_reason'],
    'stopped_round': result['stopped_round'],
    'final_avg_normalised_loss': result['final_avg_normalised_loss'],
    'final_avg_actual_loss': result['final_avg_actual_loss'],
    'final_normalised_loss_delta': result['final_normalised_loss_delta'],
    'n_clients': len(clients),
    'methods': list(clients[cid0].reconciled_forecasts.keys()),
    'output_paths': output_paths,
}
summary


{'stopped_early': True,
 'stop_reason': 'abs(avg_normalised_loss - previous_avg_normalised_loss) < 1e-05',
 'stopped_round': 72,
 'final_avg_normalised_loss': 0.38955912511877905,
 'final_avg_actual_loss': 0.15062840475331865,
 'final_normalised_loss_delta': 9.909710182165465e-06,
 'n_clients': 400,
 'methods': ['bu',
  'mint_cov',
  'mint_shrinkage',
  'mint_ols',
  'mint_var',
  'wls_structure'],
 'output_paths': {'forecasts': 'fh1_FHF_forecasts.csv',
  'per_node_metrics': 'fh1_FHF_per_node_metrics.csv',
  'metrics_by_level': 'fh1_FHF_metrics_by_level.csv',
  'overall_metrics': 'fh1_FHF_overall_metrics.csv',
  'round_logs': 'fh1_FHF_round_logs.csv',
  'timing': 'fh1_FHF_timing.csv'}}